# 🤖 RLHF / RLAIF Fine-Tuning with TruLens Metrics & TRL

Reinforcement Learning from Human/AI Feedback (RLHF / RLAIF) optimizes language model outputs by training policies against a scalar **reward function**.

TruLens metrics (such as Groundedness, Relevance, and Safety) are natural reward signals for policy gradient trainers (e.g. Hugging Face TRL's `GRPOTrainer` and `PPOTrainer`).

### Objective
In this notebook, we demonstrate how to:
1. Wrap TruLens feedback functions into RL reward signals using `TRLRewardAdapter` and `RewardFunction`.
2. Apply linear score transformations (e.g. mapping $[0, 1]$ feedback scores into $[-1, 1]$ reward signals).
3. Connect TruLens reward functions directly into a TRL training loop (`GRPOTrainer` / `PPOTrainer`).


In [ ]:
import os
import pandas as pd
from trulens.apps.rl import TRLRewardAdapter, RewardFunction

# Setup demonstration feedback function (e.g. Groundedness or Relevance)
def mock_groundedness_feedback(prompt: str, response: str, **kwargs) -> float:
    # Simulates groundedness score between 0.0 and 1.0
    if "unsupported" in response.lower() or "hallucinated" in response.lower():
        return 0.1
    return 0.95

# 1. Create TRL Reward Adapter with 2x - 1 scaling ([0, 1] -> [-1, 1])
reward_adapter = TRLRewardAdapter(
    feedback_fn=mock_groundedness_feedback,
    transform="2x-1"
)

print("TRLRewardAdapter initialized successfully.")


## 🧪 Batch Reward Evaluation

TRL trainers pass batches of generated prompts and candidate completions to `reward_funcs`. The adapter evaluates each pair and returns a list of float rewards.

In [ ]:
prompts = [
    "Explain quantum computing in simple terms.",
    "What is the capital of France?"
]

completions = [
    "Quantum computing uses qubits and superposition to process information.",
    "The capital of France is Paris."
]

rewards = reward_adapter(prompts=prompts, completions=completions)
for p, c, r in zip(prompts, completions, rewards):
    print(f"Prompt: {p}
Completion: {c}
Reward: {r:+.2f}
" + "-"*40)


## 🚀 Integrating with Hugging Face TRL GRPOTrainer

`TRLRewardAdapter` is directly compatible with TRL's `GRPOTrainer` / `PPOTrainer` via the `reward_funcs` parameter:

```python
from trl import GRPOTrainer, GRPOConfig

# Pass reward_adapter directly into GRPOTrainer
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_adapter],
    train_dataset=dataset,
    args=GRPOConfig(output_dir="./results", per_device_train_batch_size=2)
)
# trainer.train()
```
